# Hands-on 29 (Bab 11): Mini Proyek Klasifikasi Sederhana di Raspberry Pi / Emulator

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Mini proyek end-to-end meniru deployment edge: simulator sensor -> model lokal
(joblib dari Hands-on 27) -> status klasifikasi -> log CSV dan/atau MQTT.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Mini Proyek tanpa MQTT (simulator -> model -> log CSV)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/edge_room_classifier_project.py", "--steps", "15", "--delay", "0"], capture_output=True, text=True)
print(r.stdout)

=== Mini proyek edge room classifier dimulai ===
step=0 | T=29.75 H=40.75 CO2=1013 P=148.98 -> status=warning
step=1 | T=25.26 H=43.07 CO2=628 P=218.1 -> status=normal
step=2 | T=24.29 H=42.81 CO2=926 P=201.07 -> status=normal
step=3 | T=24.24 H=45.97 CO2=1309 P=144.09 -> status=warning
step=4 | T=29.3 H=64.28 CO2=463 P=251.76 -> status=warning
step=5 | T=25.44 H=52.68 CO2=1019 P=131.1 -> status=warning
step=6 | T=32.61 H=50.1 CO2=639 P=175.99 -> status=warning
step=7 | T=27.23 H=50.32 CO2=991 P=261.43 -> status=normal
step=8 | T=30.57 H=56.09 CO2=1225 P=115.76 -> status=warning
step=9 | T=26.64 H=58.86 CO2=1190 P=215.47 -> status=normal
step=10 | T=30.34 H=41.37 CO2=916 P=254.61 -> status=warning
step=11 | T=32.87 H=65.66 CO2=656 P=176.03 -> status=warning
step=12 | T=28.08 H=65.02 CO2=783 P=174.04 -> status=normal
step=13 | T=25.89 H=48.01 CO2=596 P=221.83 -> status=normal
step=14 | T=25.54 H=61.87 CO2=784 P=192.45 -> status=normal
[SUKSES] Log hasil inferensi disimpan ke: outputs/re

## 2. Verifikasi Log CSV

In [3]:
import pandas as pd
log = pd.read_csv("outputs/reports/edge_room_predictions.csv")
print(log.head(10).to_string(index=False))
print("\nDistribusi status:", log["predicted_status"].value_counts().to_dict())

 step  temperature_c  humidity_pct  co2_ppm  power_watt predicted_status
    0          29.75         40.75     1013      148.98          warning
    1          25.26         43.07      628      218.10           normal
    2          24.29         42.81      926      201.07           normal
    3          24.24         45.97     1309      144.09          warning
    4          29.30         64.28      463      251.76          warning
    5          25.44         52.68     1019      131.10          warning
    6          32.61         50.10      639      175.99          warning
    7          27.23         50.32      991      261.43           normal
    8          30.57         56.09     1225      115.76          warning
    9          26.64         58.86     1190      215.47           normal

Distribusi status: {'warning': 8, 'normal': 7}


## 3. Versi dengan MQTT (broker lokal)

In [4]:
import time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/mqtt/subscribe_sensor.py",
         "--topic", "iotlab/room01/prediction", "--max-messages", "6", "--timeout", "25"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    pub = subprocess.run([sys.executable, "scripts/edge_room_classifier_mqtt.py",
                          "--steps", "6", "--delay", "0"], capture_output=True, text=True)
    print(pub.stdout)
    out, _ = sub.communicate()
    print("--- Output subscriber ---")
    print(out[:800])
finally:
    broker.terminate()

=== Mini proyek edge room classifier + MQTT dimulai ===
step=0 | publish iotlab/room01/prediction -> status=warning
step=1 | publish iotlab/room01/prediction -> status=normal
step=2 | publish iotlab/room01/prediction -> status=normal
step=3 | publish iotlab/room01/prediction -> status=warning
step=4 | publish iotlab/room01/prediction -> status=warning
step=5 | publish iotlab/room01/prediction -> status=warning
[SUKSES] Semua prediksi dipublish ke broker.

--- Output subscriber ---
[INFO] Terhubung ke broker MQTT.
[INFO] Subscribe ke topik: iotlab/room01/prediction

=== Pesan baru diterima ===
Topik   : iotlab/room01/prediction
QoS     : 1
Payload : {"step": 0, "device_id": "room01", "temperature_c": 29.75, "humidity_pct": 40.75, "co2_ppm": 1013, "power_watt": 148.98, "predicted_status": "warning"}
[SUKSES] Payload JSON valid.
Device ID     : room01
Temperature C : 29.75
Humidity %    : 40.75
CO2 ppm       : 1013
Power watt    : 148.98
Timestamp     : None

=== Pesan baru diterima ===
T

## Penutup

Mini proyek menyatukan seluruh konsep Bab 11: training di laptop, ekspor model,
inferensi lokal per sampel, lalu hasil diteruskan ke log CSV dan/atau broker MQTT.
Prediksi yang diterbitkan ke topik `iotlab/room01/prediction` terverifikasi diterima
subscriber. Log simulasi tersimpan di `outputs/reports/edge_room_predictions.csv`.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 31 dalam urutan buku) agar dapat dijadikan panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 8. Langkah kerja yang harus dilakukan pembaca

Tahap 1
Latih model di laptop Ubuntu.
Tahap 2
Ekspor model:
- joblib untuk model klasik
- .tflite untuk model Keras ringan
Tahap 3
Uji inferensi lokal satu per satu.
Tahap 4
Simulasikan aliran data seolah-olah dari sensor.
Tahap 5
Jalankan mini proyek di Raspberry Pi atau emulator Linux.

Tahap 6
Tambahkan output:

terminal
CSV
MQTT
atau dashboard

Tahap 7
Evaluasi apakah model:

cukup cepat
cukup stabil
cukup ringan
dan layak untuk edge

### 3. Hasil inferensi

label prediksi
skor anomali
confidence sederhana
timestamp inferensi
versi model

12.5.4 Praktik minimum yang disarankan
Bila ingin tetap sederhana:
- simpan data mentah dan hasil inferensi saja di SQLite atau InfluxDB
- simpan log debug ke CSV
- jangan terlalu dini membuat storage terlalu kompleks

12.6 Batch retraining sederhana
12.6.1 Mengapa retraining perlu dibahas?
Banyak sistem demo berhenti setelah model pertama berhasil dijalankan. Padahal
dalam sistem nyata:

distribusi data bisa berubah,
perangkat bisa menua,
kebiasaan penggunaan berubah,
ruangan berganti pola,
mesin makin aus,
beban energi berubah.

Maka model yang awalnya bagus bisa menurun performanya. Drift data dan
penurunan performa model juga ditetapkan sebagai topik penting di bab
operasional berikutnya.

12.6.2 Apa itu batch retraining sederhana?
Untuk konteks buku ini, batch retraining sederhana berarti:

kumpulkan data baru selama periode tertentu,
gabungkan dengan data historis yang relevan,
latih ulang model secara berkala,
simpan model versi baru,
lalu gunakan model baru itu untuk inferensi berikutnya.

12.6.3 Kapan re-training dilakukan?
Contoh ritme yang realistis:

harian untuk eksperimen cepat
mingguan untuk smart room
mingguan / bulanan untuk energi
tergantung volume data dan perubahan pola

12.6.4 Mengapa disebut “batch”?
Karena pembaruan model tidak dilakukan setiap kali satu data masuk. Data
dikumpulkan dulu, lalu dilatih ulang sekaligus. Ini jauh lebih realistis untuk buku
praktis dibanding online learning penuh yang lebih kompleks.

12.6.5 Strategi yang disarankan
Untuk buku ini, strategi sederhana yang masuk akal:
- simpan data hasil inferensi ke storage
- setiap periode tertentu, jalankan skrip retraining

- evaluasi model baru pada holdout terbaru
- jika layak, simpan sebagai model aktif
Dengan ini, pembaca belajar bahwa pipeline end-to-end tidak berhenti pada
inferensi, tetapi juga punya siklus penyegaran model.

12.7 Inferensi otomatis
12.7.1 Apa maksud inferensi otomatis?
Inferensi otomatis berarti begitu data sensor masuk, sistem:

memproses data,
membentuk fitur,
memanggil model,
menghasilkan prediksi,
lalu menyimpan atau meneruskan hasil
tanpa campur tangan manual.

Inilah titik di mana pipeline benar-benar terasa “hidup”.

12.7.2 Alur inferensi otomatis yang sehat
1. data baru diterima dari MQTT
2. payload divalidasi
3. fitur dibentuk
4. model aktif dimuat atau sudah berada di memori
5. prediksi dilakukan
6. hasil ditulis ke storage
7. hasil dapat dikirim ke dashboard/alert

12.7.3 Hal yang perlu diperhatikan
Agar inferensi otomatis stabil:

model jangan dibuka-tutup terus setiap record jika tidak perlu
fitur harus konsisten dengan saat training
penamaan kolom harus stabil
kalau ada data kurang, tentukan strategi fallback
log error harus disimpan

12.7.4 Inferensi otomatis untuk use case buku ini
Smart room
- prediksi nyaman / tidak_nyaman
- atau skor anomali ruangan
Predictive maintenance

- skor anomali getaran
- atau klasifikasi normal / warning / fault
Smart energy
- prediksi beban berikutnya
- atau alert beban tidak normal

12.8 Dashboard / alert
12.8.1 Mengapa hasil model harus diteruskan ke dashboard?
Karena model tanpa output yang dapat dibaca operator akan berhenti menjadi
eksperimen laboratorium. Agar berguna, hasil inferensi harus:

terlihat,
dapat ditinjau,
bisa memicu aksi,
dan tercatat.

Itulah mengapa dashboard dan alert adalah bagian inti dari pipeline end-to-end.

12.8.2 Dashboard yang disarankan
Dengan stack buku ini, Grafana tetap menjadi pilihan utama untuk dashboard. Data
dari InfluxDB atau storage lain dapat dipakai untuk menampilkan:

data sensor mentah
hasil klasifikasi
skor anomali
histori prediksi
kondisi terakhir perangkat

12.8.3 Alert sederhana yang realistis
Untuk buku praktis, alert tidak perlu terlalu rumit. Cukup misalnya:

kirim log ke topik MQTT alert
tulis record khusus ke tabel alert
tampilkan panel merah di Grafana
cetak warning di terminal
kirim notifikasi sederhana ke Telegram/WA jika pembaca ingin
mengembangkan lebih lanjut

12.8.4 Kapan alert dipicu?
Contoh:

skor anomali > ambang
status prediksi = fault
suhu > batas aman
prediksi energi > threshold operasional
inferensi gagal berulang kali

12.8.5 Prinsip penting
Alert harus berguna, bukan berisik.
Kalau terlalu sering salah, operator akan berhenti percaya.

12.9 Docker opsional
12.9.1 Mengapa Docker dibahas sebagai opsional?
Karena Docker sangat berguna untuk:

memudahkan setup ulang
menjaga versi layanan konsisten
memindahkan sistem antar mesin
membuat eksperimen lebih repeatable

Namun, untuk pembaca pemula, Docker tidak harus menjadi syarat wajib sejak awal.
Docker ditempatkan sebagai opsional untuk reproducibility, bukan utama.

12.9.2 Kapan Docker sangat membantu?
Gunakan Docker jika:

ingin menjalankan Mosquitto, InfluxDB, dan Grafana cepat
ingin setup yang mudah diulang
ingin berbagi proyek ke teman atau mahasiswa lain
ingin menghindari konflik instalasi lokal

12.9.3 Kapan belum perlu?
Belum perlu jika:
- pembaca masih fokus memahami logika pipeline
- semua komponen sudah jalan stabil secara lokal
- targetnya pembelajaran Python dan MQTT dulu

12.9.4 Sikap yang sehat
Pahami alur sistem dulu, baru bungkus dengan Docker.
Jangan sampai container membuat pembaca hafal docker compose up tetapi
belum paham data mengalir ke mana.

12.10 Contoh arsitektur end-to-end yang bisa diimplementasikan
12.10.1 Smart Room Monitoring
Alur:

simulator atau sensor mengirim suhu, kelembaban, CO₂
Mosquitto menerima data
subscriber Python melakukan preprocessing
model klasifikasi atau anomaly detection dijalankan
hasil disimpan ke InfluxDB
Grafana menampilkan ruangan normal/warning/fault
jika perlu, alert dikirim ke topik khusus

12.10.2 Predictive Maintenance Sederhana
Alur:

data arus/getaran masuk
Python membentuk fitur
model anomaly detection memberi skor
hasil disimpan
dashboard menampilkan histori anomali
alert muncul jika skor tinggi

12.10.3 Smart Energy Analytics
Alur:

data daya/arus dikirim periodik
subscriber Python menulis data ke storage
model forecasting memprediksi beban berikutnya
dashboard menampilkan aktual vs prediksi
alert muncul jika beban tidak normal

12.11 Struktur folder proyek yang disarankan
Agar pembaca bisa cepat produktif, berikut struktur folder yang sangat masuk akal:
iot-ml-project/
├── data/
│
├── raw/
│
├── processed/
│
└── models/
├── scripts/
│
├── publisher/
│
├── subscriber/
│
├── preprocessing/
│
├── inference/
│
└── retraining/
├── dashboards/
├── logs/
├── docker/
├── requirements.txt
└── README.md

Kenapa struktur ini baik?
Karena:

data mentah dan olahan dipisah
model tidak bercampur dengan skrip
komponen pipeline jelas
memudahkan pembaca berkembang ke proyek akhir buku

12.12 Langkah kerja yang harus dilakukan pembaca
Agar bab ini benar-benar operasional, pembaca dapat mengikuti tahapan berikut.

Tahap 1 — Tentukan use case
Pilih salah satu:
- smart room
- predictive maintenance
- smart energy

Tahap 2 — Tentukan arsitektur minimal
Gunakan:

sensor/simulator
MQTT broker
Python subscriber
storage
dashboard

Tahap 3 — Implementasikan ingest dari MQTT
Pastikan data:
- terbaca
- tervalidasi
- konsisten formatnya

Tahap 4 — Tambahkan preprocessing
Lakukan:

parsing
pemilihan fitur
imputasi ringan bila perlu
normalisasi jika model butuh

Tahap 5 — Jalankan inferensi otomatis
Gunakan model yang sudah dipilih dari bab sebelumnya.

Tahap 6 — Simpan hasil
Tulis:
- data mentah
- hasil inferensi
- log error

Tahap 7 — Hubungkan ke dashboard / alert
Buat panel untuk:

data sensor
skor model
status prediksi
alarm

Tahap 8 — Tambahkan batch retraining sederhana
Jadwalkan re-training berkala dari data historis baru.

Tahap 9 — Bungkus dengan Docker bila perlu
Lakukan ini setelah logika sistem benar-benar dipahami.

12.13 Sikap profesional dalam membangun pipeline end-to-end
Ada satu pelajaran penting yang perlu ditekankan: pipeline yang baik bukan yang
paling rumit, tetapi yang paling stabil dan dapat dipelihara.
Dalam praktik IoT-ML:

sistem yang terlalu banyak komponen bisa rapuh,
model yang bagus tetapi pipeline rapuh tetap tidak berguna,
dashboard yang indah tetapi data tidak valid akan menyesatkan,
alert yang terlalu sering salah akan diabaikan.

Karena itu, untuk tahap ini:
- lebih baik pipeline kecil tetapi utuh,
- daripada banyak komponen tetapi tidak selesai terhubung.

12.14 Penutup bab
Bab 12 adalah titik ketika seluruh isi buku mulai menyatu menjadi sistem
operasional yang benar-benar hidup. Di sini pembaca belajar bahwa pipeline
IoT-ML end-to-end berarti menyusun alur lengkap dari sensor ke prediksi: arsitektur
proyek yang jelas, ingest dari MQTT ke Python, penyimpanan yang rapi, inferensi
otomatis, dashboard/alert yang berguna, dan retraining sederhana agar model tidak
membeku di masa lalu. Docker kemudian hadir sebagai opsi untuk membuat
semuanya lebih mudah diulang, tetapi bukan pengganti pemahaman sistem.